In [13]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.9 MB/s eta 0:00:00


In [1]:
!pip install sentence-transformers

!pip install -U google-generativeai langchain-google-genai

In [2]:
!pip install -q --upgrade langchain langchain-google-genai langchain-community
!pip install -q --upgrade chromadb pypdf sentence-transformers

In [26]:
from langchain_community.document_loaders import WebBaseLoader

who_urls = [
    # Infectious Diseases
    "https://www.who.int/news-room/fact-sheets/detail/malaria",
    "https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue",
    "https://www.who.int/news-room/fact-sheets/detail/tuberculosis",
    "https://www.who.int/news-room/fact-sheets/detail/hiv-aids",
    "https://www.who.int/news-room/fact-sheets/detail/cholera",
    "https://www.who.int/news-room/fact-sheets/detail/rabies",
    "https://www.who.int/news-room/fact-sheets/detail/influenza-(seasonal)",
    "https://www.who.int/news-room/fact-sheets/detail/hepatitis-a",
    "https://www.who.int/news-room/fact-sheets/detail/hepatitis-b",
    "https://www.who.int/news-room/fact-sheets/detail/hepatitis-c",
    "https://www.who.int/news-room/fact-sheets/detail/hepatitis-d",
    "https://www.who.int/news-room/fact-sheets/detail/hepatitis-e",
    "https://www.who.int/news-room/fact-sheets/detail/measles",
    "https://www.who.int/news-room/fact-sheets/detail/meningitis",
    "https://www.who.int/news-room/fact-sheets/detail/plague",
    "https://www.who.int/news-room/fact-sheets/detail/poliomyelitis",
    "https://www.who.int/news-room/fact-sheets/detail/tetanus",
    "https://www.who.int/news-room/fact-sheets/detail/typhoid",
    "https://www.who.int/news-room/fact-sheets/detail/yellow-fever",
    "https://www.who.int/news-room/fact-sheets/detail/ebola-virus-disease",
    "https://www.who.int/news-room/fact-sheets/detail/mpox",
    "https://www.who.int/news-room/fact-sheets/detail/leprosy",
    "https://www.who.int/news-room/fact-sheets/detail/diphtheria",
    "https://www.who.int/news-room/fact-sheets/detail/hantavirus",
    "https://www.who.int/news-room/fact-sheets/detail/nipah-virus",
    "https://www.who.int/news-room/fact-sheets/detail/zika-virus",
    "https://www.who.int/news-room/fact-sheets/detail/chikungunya",
    "https://www.who.int/news-room/fact-sheets/detail/leishmaniasis",
    "https://www.who.int/news-room/fact-sheets/detail/schistosomiasis",
    "https://www.who.int/news-room/fact-sheets/detail/lymphatic-filariasis",

    # Chronic / Non-Communicable Diseases
    "https://www.who.int/news-room/fact-sheets/detail/diabetes",
    "https://www.who.int/news-room/fact-sheets/detail/cancer",
    "https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)",
    "https://www.who.int/news-room/fact-sheets/detail/hypertension",
    "https://www.who.int/news-room/fact-sheets/detail/asthma",
    "https://www.who.int/news-room/fact-sheets/detail/chronic-obstructive-pulmonary-disease-(copd)",
    "https://www.who.int/news-room/fact-sheets/detail/obesity-and-overweight",
    "https://www.who.int/news-room/fact-sheets/detail/epilepsy",
    "https://www.who.int/news-room/fact-sheets/detail/dementia",
    "https://www.who.int/news-room/fact-sheets/detail/blindness-and-visual-impairment",
    "https://www.who.int/news-room/fact-sheets/detail/deafness-and-hearing-loss",
    "https://www.who.int/news-room/fact-sheets/detail/anaemia",
    "https://www.who.int/news-room/fact-sheets/detail/endometriosis",
    "https://www.who.int/news-room/fact-sheets/detail/cervical-cancer",
    "https://www.who.int/news-room/fact-sheets/detail/breast-cancer",
    "https://www.who.int/news-room/fact-sheets/detail/lung-cancer",
    "https://www.who.int/news-room/fact-sheets/detail/colorectal-cancer",
    "https://www.who.int/news-room/fact-sheets/detail/oral-health",
    "https://www.who.int/news-room/fact-sheets/detail/sickle-cell-disease",

    # Mental Health
    "https://www.who.int/news-room/fact-sheets/detail/mental-disorders",
    "https://www.who.int/news-room/fact-sheets/detail/depression",
    "https://www.who.int/news-room/fact-sheets/detail/suicide",
    "https://www.who.int/news-room/fact-sheets/detail/autism-spectrum-disorders",
    "https://www.who.int/news-room/fact-sheets/detail/schizophrenia",

    # Respiratory
    "https://www.who.int/news-room/fact-sheets/detail/pneumonia",
    "https://www.who.int/news-room/fact-sheets/detail/diarrhoeal-disease",

    # Headache and Pain
    "https://www.who.int/news-room/fact-sheets/detail/headache-disorders",

    # Mother and Child
    "https://www.who.int/news-room/fact-sheets/detail/infant-and-young-child-feeding",
    "https://www.who.int/news-room/fact-sheets/detail/preterm-birth",
    "https://www.who.int/news-room/fact-sheets/detail/maternal-mortality",
    "https://www.who.int/news-room/fact-sheets/detail/adolescent-pregnancy",
    "https://www.who.int/news-room/fact-sheets/detail/child-maltreatment",
    "https://www.who.int/news-room/fact-sheets/detail/malnutrition",

    # Environment and Lifestyle
    "https://www.who.int/news-room/fact-sheets/detail/ambient-(outdoor)-air-quality-and-health",
    "https://www.who.int/news-room/fact-sheets/detail/drinking-water",
    "https://www.who.int/news-room/fact-sheets/detail/food-safety",
    "https://www.who.int/news-room/fact-sheets/detail/physical-activity",
    "https://www.who.int/news-room/fact-sheets/detail/tobacco",
    "https://www.who.int/news-room/fact-sheets/detail/alcohol",
    "https://www.who.int/news-room/fact-sheets/detail/climate-change-and-health",
    "https://www.who.int/news-room/fact-sheets/detail/household-air-pollution-and-health",
    "https://www.who.int/news-room/fact-sheets/detail/sanitation",
    "https://www.who.int/news-room/fact-sheets/detail/ultraviolet-radiation",
    "https://www.who.int/news-room/fact-sheets/detail/radiation-and-health",

    # Vaccination and Prevention
    "https://www.who.int/news-room/fact-sheets/detail/immunization-coverage",
    "https://www.who.int/news-room/fact-sheets/detail/vaccines-and-immunization",
    "https://www.who.int/news-room/fact-sheets/detail/antimicrobial-resistance",
    "https://www.who.int/news-room/fact-sheets/detail/antibiotic-resistance",

    # Injuries and Violence
    "https://www.who.int/news-room/fact-sheets/detail/road-traffic-injuries",
    "https://www.who.int/news-room/fact-sheets/detail/burns",
    "https://www.who.int/news-room/fact-sheets/detail/drowning",
    "https://www.who.int/news-room/fact-sheets/detail/falls",
    "https://www.who.int/news-room/fact-sheets/detail/violence-against-women",
    "https://www.who.int/news-room/fact-sheets/detail/elder-abuse",
    "https://www.who.int/news-room/fact-sheets/detail/female-genital-mutilation",

    # STIs
    "https://www.who.int/news-room/fact-sheets/detail/sexually-transmitted-infections-(stis)",
    "https://www.who.int/news-room/fact-sheets/detail/human-papillomavirus-(hpv)-and-cervical-cancer",

    # Other Important Topics
    "https://www.who.int/news-room/fact-sheets/detail/ageing-and-health",
    "https://www.who.int/news-room/fact-sheets/detail/disability-and-health",
    "https://www.who.int/news-room/fact-sheets/detail/patient-safety",
    "https://www.who.int/news-room/fact-sheets/detail/palliative-care",
    "https://www.who.int/news-room/fact-sheets/detail/noncommunicable-diseases",
    "https://www.who.int/news-room/fact-sheets/detail/arsenic",
    "https://www.who.int/news-room/fact-sheets/detail/lead-poisoning",
    "https://www.who.int/news-room/fact-sheets/detail/mercury-and-health",
]

print(f"Loading {len(who_urls)} WHO fact sheets...")
print("This will take 3-5 minutes...\n")

loader = WebBaseLoader(who_urls)
who_documents = loader.load()

# Remove any empty documents
who_documents = [doc for doc in who_documents if len(doc.page_content) > 100]

print(f"\nSuccessfully loaded {len(who_documents)} WHO fact sheets!")

Loading 95 WHO fact sheets...
This will take 3-5 minutes...


Successfully loaded 95 WHO fact sheets!


In [3]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate

In [14]:
import os
from google.colab import userdata

# Set your Gemini API key
os.environ["GROQ_API_KEY"] = "Your_API_KEY"
# Configuration
PDF_DIRECTORY = "data/"
CHROMA_DB_PATH = "chroma_db/"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K_RESULTS = 5

In [5]:
# Upload WHO PDFs
!mkdir -p data

from google.colab import files
uploaded = files.upload()

import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"data/{filename}")
    print(f"Moved {filename} to data/")

Saving report_1.pdf to report_1.pdf
Saving report_2.pdf to report_2.pdf
Saving report_3.pdf to report_3.pdf
Saving report_4.pdf to report_4.pdf
Saving report_5.pdf to report_5.pdf
Saving report_6.pdf to report_6.pdf
Saving report_7.pdf to report_7.pdf
Saving report_8.pdf to report_8.pdf
Saving report_9.pdf to report_9.pdf
Saving report_10.pdf to report_10.pdf
Saving report_11.pdf to report_11.pdf
Saving report_12.pdf to report_12.pdf
Saving report_13.pdf to report_13.pdf
Saving report_14.pdf to report_14.pdf
Saving report_15.pdf to report_15.pdf
Saving report_16.pdf to report_16.pdf
Saving report_17.pdf to report_17.pdf
Saving report_18.pdf to report_18.pdf
Saving report_20.pdf to report_20.pdf
Saving report_23.pdf to report_23.pdf
Saving report_24.pdf to report_24.pdf
Saving report_25.pdf to report_25.pdf
Moved report_1.pdf to data/
Moved report_2.pdf to data/
Moved report_3.pdf to data/
Moved report_4.pdf to data/
Moved report_5.pdf to data/
Moved report_6.pdf to data/
Moved report_7

In [27]:
all_documents = documents + who_documents
print(f"Total: {len(documents)} PDF pages + {len(who_documents)} WHO web pages = {len(all_documents)}")

all_chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks: {len(all_chunks)}")

Total: 1001 PDF pages + 95 WHO web pages = 1096
Total chunks: 9199


In [31]:
import shutil
if os.path.exists("chroma_db_v2"):
    shutil.rmtree("chroma_db_v2")

vector_db = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="chroma_db_v2/"
)
print(f"Vector DB created with {len(all_chunks)} chunks!")

Vector DB created with 9199 chunks!


In [32]:
test_queries = [
    "symptoms of malaria",
    "how to prevent diabetes",
    "symptoms of headache",
    "what causes depression",
    "drinking water guidelines"
]

for q in test_queries:
    results = vector_db.similarity_search(q, k=1)
    print(f"Q: {q}")
    print(f"A: {results[0].page_content[:150]}...")
    print(f"Source: {results[0].metadata.get('source', 'WHO Web')}")
    print("-" * 50)

Q: symptoms of malaria
A: . The other malaria species which can infect humans are P. malariae, P. ovale and P. knowlesi.SymptomsThe most common early symptoms of malaria are fe...
Source: https://www.who.int/news-room/fact-sheets/detail/malaria
--------------------------------------------------
Q: how to prevent diabetes
A: . In addition, around 11% of cardiovascular deaths were caused by high blood glucose.A healthy diet, regular physical activity, maintaining a normal b...
Source: https://www.who.int/news-room/fact-sheets/detail/diabetes
--------------------------------------------------
Q: symptoms of headache
A: . It can be triggered by alcohol, sleep disruption and certain foods.Attacks typically include headache, which is often:of moderate or severe intensit...
Source: https://www.who.int/news-room/fact-sheets/detail/headache-disorders
--------------------------------------------------
Q: what causes depression
A: . It involves a depressed mood or loss of pleasure or interest in 

In [34]:
# Cell F — Rebuild RAG chain with full data
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

PROMPT_TEMPLATE = """
You are an AI Health Assistant powered by WHO documents.

Answer ONLY from the provided context.
If the context does not contain enough information, say:
"I don't have enough information from WHO documents to answer this."

Always mention the source document.
Do not provide medical diagnoses.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready with full WHO data!")

RAG chain ready with full WHO data!


In [35]:
# Cell G — Query function
def ask(question):
    print(f"Question: {question}")
    print("=" * 50)
    response = rag_chain.invoke(question)
    print(f"\nAnswer:\n{response}")
    print("=" * 50)

In [36]:
# Cell J — Test: Headache
ask("What causes headaches?")

Question: What causes headaches?

Answer:
According to the WHO fact sheet on migraine and other headache disorders (https://www.who.int/news-room/fact-sheets/detail/headache-disorders), headaches can be caused by or occur secondarily to a long list of other conditions, the most common of which is medication-overuse headache. The exact cause of migraine is currently unknown but it is thought to result from the release of pain producing inflammatory substances around the nerves and blood vessels of the head. Additionally, headaches can be triggered by alcohol, sleep disruption, and certain foods.
